<a href="https://colab.research.google.com/github/moubanidutta05-del/PGII_MOUBANI/blob/main/Zonal_Statistics_Visualization_using_MODIS_Time_Series_%26_District_Shapefiles_of_India.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee

In [2]:
import geemap

In [3]:
ee. Authenticate()

True

In [4]:
ee.Initialize(project="moubani-ee")

In [5]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [6]:
kolkata = ee.FeatureCollection('projects/moubani-ee/assets/Taluk_Boundary')
map.addLayer(kolkata,{},'kolkata')
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [7]:
tree_ic = ee.ImageCollection("MODIS/006/MOD44B") \
    .filterBounds(kolkata) \
    .select('Percent_Tree_Cover')

In [8]:
tree_early = tree_ic.filterDate('2001-01-01','2003-12-31').mean().clip(kolkata)

In [9]:
tree_recent = tree_ic.filterDate('2018-01-01','2020-12-31').mean().clip(kolkata)

In [10]:
lai_ic = ee.ImageCollection("MODIS/061/MCD15A3H") \
    .filterBounds(kolkata) \
    .select('Lai')

In [11]:
lai_early = lai_ic.filterDate('2001-01-01','2003-12-31').mean().clip(kolkata)

In [12]:
lai_recent = lai_ic.filterDate('2018-01-01','2020-12-31').mean().clip(kolkata)

In [13]:
lst_ic = ee.ImageCollection("MODIS/061/MOD11A1") \
    .filterBounds(kolkata) \
    .select('LST_Day_1km')

In [14]:
def to_celsius(img):
    return img.multiply(0.02).subtract(273.15)

In [15]:
lst_ic = lst_ic.map(to_celsius)

In [16]:
lst_early = lst_ic.filterDate('2002-01-01','2003-12-31').mean().clip(kolkata)

In [17]:
lst_recent = lst_ic.filterDate('2019-01-01','2020-12-31').mean().clip(kolkata)

In [18]:
def zonal(image):
    return image.reduceRegions(
        collection=kolkata,
        reducer=ee.Reducer.mean(),
        scale=1000
    )

In [19]:
tree_early_stats = zonal(tree_early)
tree_recent_stats = zonal(tree_recent)

In [20]:
lai_early_stats = zonal(lai_early)
lai_recent_stats = zonal(lai_recent)

In [21]:
lst_early_stats = zonal(lst_early)
lst_recent_stats = zonal(lst_recent)

In [22]:
Map = geemap.Map()

In [23]:
Map.addLayer(tree_early, {'min':0,'max':100}, 'Tree Early Kolkata')
Map.addLayer(tree_recent, {'min':0,'max':100}, 'Tree Recent Kolkata')

In [24]:
Map.addLayer(lai_early, {'min':0,'max':6}, 'LAI Early Kolkata')
Map.addLayer(lai_recent, {'min':0,'max':6}, 'LAI Recent Kolkata')


In [25]:
Map.addLayer(lst_early, {'min':20,'max':40}, 'LST Early Kolkata')
Map.addLayer(lst_recent, {'min':20,'max':40}, 'LST Recent Kolkata')

In [26]:
Map.addLayer(kolkata, {}, 'Boundary')

In [27]:
geemap.ee_export_vector(tree_early_stats, filename='kolkata_tree_early.geojson')
geemap.ee_export_vector(tree_recent_stats, filename='kolkata_tree_recent.geojson')


Generating URL ...
Please wait ...
Data downloaded to /content/kolkata_tree_early.geojson
Generating URL ...
Please wait ...
Data downloaded to /content/kolkata_tree_recent.geojson


In [28]:
geemap.ee_export_vector(lai_early_stats, filename='kolkata_lai_early.geojson')
geemap.ee_export_vector(lai_recent_stats, filename='kolkata_lai_recent.geojson')

Generating URL ...
Please wait ...
Data downloaded to /content/kolkata_lai_early.geojson
Generating URL ...
Please wait ...
Data downloaded to /content/kolkata_lai_recent.geojson


In [32]:
lst_col = ee.ImageCollection("MODIS/061/MOD11A1").select("LST_Day_1km")

In [33]:
lst_early = (
    lst_col
    .filterBounds(kolkata)
    .filterDate("2001-01-01","2003-12-31")
    .mean()
    .multiply(0.02)
    .subtract(273.15)
    .rename("LST")
    .clip(kolkata)
)

In [34]:
lst_recent = (
    lst_col
    .filterBounds(kolkata)
    .filterDate("2023-01-01","2024-12-31")
    .mean()
    .multiply(0.02)
    .subtract(273.15)
    .rename("LST")
    .clip(kolkata)
)

In [35]:
def zonal(image):
    return image.reduceRegions(
        collection=ee.FeatureCollection([ee.Feature(kolkata)]),
        reducer=ee.Reducer.mean(),
        scale=1000
    )


In [36]:
geemap.ee_export_vector(lst_early_stats, filename='LST_early.geojson')
geemap.ee_export_vector(lst_recent_stats, filename='LST_recent.geojson')

EEException: Image.reduceRegions: Image has no bands.